# ⚡ 15 · FP4/NVFP4 量化与 2:4 结构化稀疏（Blackwell/Vera Rubin 时代）

> 本节承接 [07 量化与模型压缩](07-量化与模型压缩PTQ-GPTQ-AWQ-SmoothQuant.ipynb)（PTQ / GPTQ / AWQ / SmoothQuant）与 [02 混合精度与显存账本](02-混合精度与显存账本.ipynb)：
> 当权重从 BF16 → FP8 一路压到 4 bit，**再往下就只能靠两样东西** —— 更聪明的**缩放**（block scale / NVFP4）与硬件原生的**稀疏**（2:4 structured sparsity）。
>
> 一句话结论：**FP4 不是"更小的 INT4"，它是浮点数** —— 用 2 位指数换动态范围、用 1 位尾数换精度，再用"每 16 个权重共用一把尺"把丢掉的那点精度补回来。

## 📋 本节要解决的问题

| # | 问题 | 一句话答案 | 本节的量化口径 |
|---|---|---|---|
| 1 | FP4 到底能表示哪些数？ | 只有 16 个刻度：$0,\ \pm0.5,\ \pm1,\ \pm1.5,\ \pm2,\ \pm3,\ \pm4,\ \pm6$ | E2M1 位布局（1 符号 + 2 指数 + 1 尾数） |
| 2 | 为什么不用 INT4？ | 浮点有 2 位指数，**相对精度在小值附近更细**；INT4 是等差刻度 | 同一分布下 MSE / 余弦相似度实测 |
| 3 | block scale 解决什么？ | 量化单位从"整张量一把尺"换成"每 16 个权重一把尺"，抵消 16 个刻度的粗糙 | block ∈ {8, 16, 32, 64, 128} 扫描 |
| 4 | scale 自己被量化会怎样？ | NVFP4 把 scale 存成 FP8 E4M3，**scale 自带几个百分点误差**，是第三条误差来源 | 误差分解实验（数值 / scale / 截断） |
| 5 | 2:4 稀疏为什么能加速？ | 每 4 个权重只留 2 个 → Tensor Core 有原生 2:4 通路，**MAC 数减半** | 稀疏度 / 保留能量 / MSE 实测 |
| 6 | 稀疏和量化能叠加吗？ | 能，但**先剪枝还是先量化**结果不同，需要实测取舍 | 组合顺序实验 |
| 7 | 2026 年这些落到哪个硬件？ | Blackwell 原生 NVFP4；Vera Rubin 上出现 FP4 + 2:4 的联合量化方案 | 前沿对照表（只做定性） |

## 1️⃣ 直觉与大白话

> 💡 **FP4 = 一把只有 8 格刻度的尺子**
> BF16 是一把刻度很密的尺，量 5.2 能读到 5.1875；FP4 的正刻度只有 `0.5, 1, 1.5, 2, 3, 4, 6` 这 7 格，
> 量 5.2 只能记成 4 或 6 —— 相对误差 15%~23%。这就是"4 bit 只够记 16 个数"的直接后果。

> 💡 **block scale = 每 16 个权重共用一把盒尺**
> 一把尺量一群大小差很多的书，小的全被记成 0。但如果**每 16 本书单独配一把按这批最大值定标的盒尺**，
> 绝对范围就自适应了：这批最大是 0.02，那刻度就是 `0.0017, 0.0033, …, 0.02`。**相对刻度不变，绝对量程随块走。**

> 💡 **2:4 稀疏 = 每 4 个座位只坐 2 个人**
> 每 4 个连续权重里只保留绝对值最大的 2 个，另外 2 个置零。硬件（Tensor Core）可以直接按"每 4 读 2"的方式取数，
> GEMM 的乘加次数就只剩一半 —— 但这要求**存储格式也按 2:4 摆放**，否则省不下带宽。

一条主线：**量化解决"每个数存多小"，稀疏解决"存几个数"，block scale 解决"一把尺不够用"。**

In [ ]:
# ---- matplotlib 中文字体（防图内中文乱码）----
import matplotlib

matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
# -------------------------------------------
import numpy as np

# ============ FP4 (E2M1) 的可表示值 ============
# 位布局：1 符号位 | 2 指数位（bias = 1）| 1 尾数位
# 指数 e ∈ {0,1,2,3} → 2^(e-1)；尾数 m ∈ {0,1} → ×(1 + m/2)，e=0 时为次正规 → ×(m/2)
FP4_POS = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0])
FP4_ALL = np.concatenate([-FP4_POS[:0:-1], FP4_POS])   # 含 0 的 15 个不同取值

print('位布局        : [符号 1][指数 2][尾数 1]   （指数 bias = 1）')
print('正刻度（7 格）: ', ', '.join('%g' % v for v in FP4_POS[1:]))
print('全部取值（15）: ', ', '.join('%g' % v for v in FP4_ALL))
print('最大可表示值  : %g' % FP4_POS.max())
print('最小正取值    : %g（次正规）' % FP4_POS[1])
print('量程比 max/min: %g 倍' % (FP4_POS.max() / FP4_POS[1]))

# ---- 朴素"最近邻"量化：把任意实数塞进这 15 个刻度 ----
def quantize(fp4_values, x):
    '''最近邻量化（不做 block scale，直接用全局刻度）。'''
    x = np.asarray(x, dtype=float)
    d = np.abs(x.reshape(-1, 1) - fp4_values.reshape(1, -1))
    return fp4_values[d.argmin(axis=1)].reshape(x.shape)

demo = np.array([5.2, 0.3, -2.7, 11.0, 0.05])
q = quantize(FP4_ALL, demo)
print('\n原始值 → FP4 最近邻（无 block scale）：')
for a, b in zip(demo, q):
    rel = abs(b - a) / max(abs(a), 1e-12)
    print('  %7.3f → %7.3f    相对误差 %6.1f%%' % (a, b, 100 * rel))

assert np.isclose(FP4_POS.max(), 6.0), 'FP4 最大值应为 6'
assert np.isclose(FP4_POS.min(), 0.0) and np.isclose(FP4_POS[1], 0.5), 'FP4 最小正取值应为 0.5'
assert np.isclose(FP4_POS.max() / FP4_POS[1], 12.0), 'FP4 量程比应为 12'
print('\n✅ 断言：FP4 只有 15 个不同取值，量程比 12；小值（0.05）在没有 block scale 时直接被压成 0 —— 这就是 block scale 存在的理由')

## 2️⃣ 为什么是 FP4 而不是 INT4：动态范围与 block scaling

### 2.1 位布局解码表（面试要能当场算出）

| 指数域 $e$ | 尾数域 $m$ | 二进制 | 值（$2^{e-1}\times(1+m/2)$，$e{=}0$ 时视为次正规 $m/2$） |
|---|---|---|---|
| 0 | 0 | `0 00 0` | 0 |
| 0 | 1 | `0 00 1` | **0.5** |
| 1 | 0 | `0 01 0` | 1.0 |
| 1 | 1 | `0 01 1` | 1.5 |
| 2 | 0 | `0 10 0` | 2.0 |
| 2 | 1 | `0 10 1` | 3.0 |
| 3 | 0 | `0 11 0` | 4.0 |
| 3 | 1 | `0 11 1` | **6.0** |

### 2.2 FP4 vs INT4：差别不在"位数"，在"刻度怎么分"

| 维度 | INT4（对称，per-channel） | FP4（E2M1，per-block scale） |
|---|---|---|
| 取值 | 16 个**等间距**整数 $-8..7$（或 $-7..7$） | 15 个**对数分布**刻度：0.5,1,1.5,2,3,4,6 |
| 小值精度 | 步长固定 = $\max/7$，小于半个步长直接归零 | 步长随数值缩小（指数控制）：0.5 附近 Schritt ≈ 0.5，6 附近 ≈ 2 |
| 动态范围 | 靠 **scale** 撑：$\max/7$ 之外全丢 | 刻度本身跨 12 倍；块内自适应把绝对范围再平移 |
| 离群点 | 一个离群点会把其余全部压扁 | 块内离群点只污染本块（16 个），其余块安全 |

> 💡 关键直觉：**INT4 的误差是"绝对步长"误差，FP4 的误差是"相对精度"误差**。
> 权重近似高斯时两者同量级；**重尾 / 含离群通道**时，浮点的相对刻度更占便宜 —— 但这要实测，不能背结论（见 §5 实验 1）。

### 2.3 block scale 的两层含义

1. **定标**：每 block 取 $\text{absmax}$，令 $\text{scale} = \text{absmax}/6$，把块内数值压进 $[-6, 6]$；
2. **代价**：NVFP4 的 scale 不是无限精度的，它自己要被存成 **FP8 E4M3**（8 bit / 16 个权重 = 0.0625 B/参数），
   于是"scale 的舍入误差"会成为第三条误差来源 —— 这一点在 GPTQ/AWQ 时代常被忽略。

In [ ]:
# ============ 误差直觉：同一重尾分布下三种 4-bit 方案 ============
rng = np.random.default_rng(20261123)

def dist_gauss(n):
    return rng.normal(0, 1, n)

def dist_heavy(n):
    '''student-t(3) 标准化到单位方差：重尾，模拟"少数极大权重"。'''
    x = rng.standard_t(3, n)
    return x / np.sqrt(3.0)

def dist_outlier(n, frac=0.01, mult=20.0):
    x = rng.normal(0, 1, n)
    k = int(n * frac)
    idx = rng.choice(n, k, replace=False)
    x[idx] *= mult
    return x

def quantize_e4m3(x):
    '''FP8 E4M3 的数值模拟：1 符号 + 4 指数（bias 7）+ 3 尾数。
    正规区：2^-6 ~ 448，步长 2^(e-3)；次正规区：步长固定 2^-9（最小次正规值）。'''
    x = np.asarray(x, dtype=float)
    sign = np.sign(x)
    a = np.abs(x)
    out = np.zeros_like(a)
    nz = a > 0
    if np.any(nz):
        aa = a[nz]
        e = np.floor(np.log2(aa))
        e_norm = np.clip(e, -6, 8)
        step = 2.0 ** (e_norm - 3)                     # 3 位尾数 → 相对分辨率 1/8
        step = np.where(e < -6, 2.0 ** -9, step)       # 次正规：固定步长
        q = np.round(aa / step) * step
        out[nz] = np.clip(q, 0.0, 448.0)
    return sign * out

def quantize_e8m0(x):
    '''MXFP4 的 scale 类型：只有指数，取值是 2 的整数次幂（无尾数）。'''
    x = np.asarray(x, dtype=float)
    a = np.abs(x)
    out = np.zeros_like(a)
    nz = a > 0
    out[nz] = 2.0 ** np.round(np.log2(a[nz]))
    return out

def quantize_int4_perchannel(w, qmax=7.0):
    w = np.asarray(w, dtype=float)
    amax = np.abs(w).max()
    if amax == 0:
        return np.zeros_like(w)
    scale = amax / qmax
    return np.clip(np.round(w / scale), -qmax, qmax) * scale

def quantize_int4_block(w, block=16, qmax=7.0):
    '''INT4 + 每 block 一个 scale（与 NVFP4 同样的粒度，做公平对照）。'''
    w = np.asarray(w, dtype=float).reshape(-1)
    n = w.size
    pad = (-n) % block
    if pad:
        w = np.concatenate([w, np.zeros(pad)])
    blk = w.reshape(-1, block)
    amax = np.abs(blk).max(axis=1, keepdims=True)
    amax = np.where(amax == 0, 1e-30, amax)
    scale = amax / qmax
    q = np.clip(np.round(blk / scale), -qmax, qmax) * scale
    return q.reshape(-1)[:n]

def quantize_fp8_perchannel(w):
    return quantize_e4m3(w)

def quantize_nvfp4(w, block=16, quant_scale=True, fp4_pos=FP4_ALL):
    '''NVFP4：每 block 一个 FP8-E4M3 scale，数值用 FP4 最近邻。'''
    w = np.asarray(w, dtype=float).reshape(-1)
    n = w.size
    pad = (-n) % block
    if pad:
        w = np.concatenate([w, np.zeros(pad)])
    blk = w.reshape(-1, block)
    amax = np.abs(blk).max(axis=1, keepdims=True)
    amax = np.where(amax == 0, 1e-30, amax)
    scale = amax / fp4_pos.max()
    if quant_scale:
        scale = quantize_e4m3(scale)            # ← scale 自己也被量化（NVFP4 的关键细节）
    scale = np.where(scale == 0, 1e-30, scale)
    norm = blk / scale
    d = np.abs(norm.reshape(-1, 1) - fp4_pos.reshape(1, -1))
    q = fp4_pos[d.argmin(axis=1)].reshape(norm.shape) * scale
    return q.reshape(-1)[:n]

def quantize_mxfp4(w, block=32, fp4_pos=FP4_ALL):
    '''MXFP4（OCP 标准）：block = 32，scale 为共享指数 E8M0（2 的整数次幂）。'''
    w = np.asarray(w, dtype=float).reshape(-1)
    n = w.size
    pad = (-n) % block
    if pad:
        w = np.concatenate([w, np.zeros(pad)])
    blk = w.reshape(-1, block)
    amax = np.abs(blk).max(axis=1, keepdims=True)
    amax = np.where(amax == 0, 1e-30, amax)
    scale = quantize_e8m0(amax / fp4_pos.max())
    scale = np.where(scale == 0, 1e-30, scale)
    norm = blk / scale
    d = np.abs(norm.reshape(-1, 1) - fp4_pos.reshape(1, -1))
    q = fp4_pos[d.argmin(axis=1)].reshape(norm.shape) * scale
    return q.reshape(-1)[:n]

def mse(a, b):
    return float(np.mean((np.asarray(a) - np.asarray(b)) ** 2))

def cos(a, b):
    a = np.asarray(a).ravel(); b = np.asarray(b).ravel()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-30))

def rel_err(a, b):
    return float(np.linalg.norm(np.asarray(a) - np.asarray(b)) / (np.linalg.norm(a) + 1e-30))

w = dist_heavy(4096)
rows = [
    ('NVFP4 (block16, FP8 scale)', quantize_nvfp4(w, 16, True)),
    ('MXFP4 (block32, E8M0)',      quantize_mxfp4(w, 32)),
    ('INT4  (block16)',            quantize_int4_block(w, 16)),
    ('INT4  (per-channel)',        quantize_int4_perchannel(w)),
    ('FP8   (E4M3)',               quantize_fp8_perchannel(w)),
]
print('重尾分布（student-t3，n=4096）下的误差：')
print('%-28s %12s %12s %10s' % ('方案', 'MSE', '相对误差', '余弦'))
for name, q in rows:
    print('%-28s %12.3e %12.4f %10.6f' % (name, mse(w, q), rel_err(w, q), cos(w, q)))

m_fp4 = mse(w, rows[0][1])
m_i4b = mse(w, rows[2][1])
m_i4pc = mse(w, rows[3][1])
m_fp8 = mse(w, rows[4][1])
assert m_fp8 < m_fp4, 'FP8 有 4 位指数 + 3 位尾数，误差应小于 FP4（同一向量）'
assert 0.2 <= m_fp4 / m_i4b <= 5.0, '同粒度下 FP4 与 INT4 应处在同一数量级（不预设谁更好）'
assert m_i4b < m_i4pc, '同样的 block 粒度应优于 per-channel（per-channel 被离群值拖垮）'
print('\n✅ 断言：FP8 误差 < FP4；同 block 粒度下 FP4/INT4 之比 %.2f（同量级）；INT4 换块粒度后显著变好' % (m_fp4 / m_i4b))

## 3️⃣ 机制图

![FP4 位布局、per-block scale 与 2:4 稀疏三重机制](images/15_fp4_sparse.svg)

读图三步：
1. **左**：`1+2+1` 的位布局决定只有 8 个正刻度 —— 尺子本身就是粗的，这是 4 bit 的物理上限；
2. **中**：每 16 个权重共用一把"按块内最大值定标"的尺（scale 再被 FP8 量化一次），把粗刻度摊到合适的绝对范围；
3. **右**：2:4 稀疏不改数值精度，而是让硬件"每 4 个位置只读 2 个"，把 MAC 数减半 —— 前提是存储格式与 Tensor Core 通路配套。

## 4️⃣ 手写实现

### 4.1 API 契约（先约定接口，再写实现）

| 函数 | 输入 | 输出 | 误差来源 |
|---|---|---|---|
| `quantize_fp4_block(w, block, quant_scale)` | 权重矩阵 / 向量 | 反量化后的浮点权重 | ① 16 刻度舍入 ② scale 量化 ③ 次正规截断 |
| `quantize_mxfp4(w, block)` | 同上 | 同上 | ① ② 同上，scale 换成 E8M0（只有 2 的幂） |
| `quantize_int4_perchannel(w)` | 同上 | 同上 | 绝对步长 + 离群点压扁 |
| `quantize_fp8_perchannel(w)` | 同上 | 同上 | 仅 3 位尾数舍入 |
| `prune_2_4(w, outlier_aware)` | 权重矩阵 | 稀疏化后的权重 | 结构约束（每 4 留 2） |

> 真正的 FP4 kernel 是位打包（两个 4-bit 塞进一个 byte）+ 查表反量化；本节用**数值模拟**（在 15 个刻度上取最近邻）代替位运算，
> 因为要考察的是**误差与收益的量级**，不是指令编码本身。

In [ ]:
# ============ 4.1 FP4 编解码 + block scale（含 scale 量化开关） ============
def fp4_encode_index(x_normalized, fp4_pos=FP4_POS):
    '''把归一化到 [-6,6] 的数值编码成 FP4 刻度下标（0..7），符号单独处理。'''
    a = np.abs(np.asarray(x_normalized, dtype=float)).reshape(-1, 1)
    d = np.abs(a - fp4_pos.reshape(1, -1))
    idx = d.argmin(axis=1)
    return idx.reshape(np.shape(x_normalized)), np.sign(x_normalized)

def fp4_decode(idx, sign, fp4_pos=FP4_POS):
    return sign * fp4_pos[idx]

def quantize_fp4_block(w, block=16, quant_scale=True):
    '''NVFP4 风格：每 block 一个 scale（= absmax/6），可选把 scale 量化到 E4M3。'''
    return quantize_nvfp4(w, block=block, quant_scale=quant_scale)

# ---- 自检 1：刻度内的值必须能无损往返 ----
signed = np.concatenate([-FP4_POS[:0:-1], FP4_POS])
idx, sgn = fp4_encode_index(signed / 6.0 * 6.0)   # 除以 6 再乘 6，等价恒等
back = fp4_decode(idx, sgn)
assert np.allclose(back, signed), 'FP4 刻度值应能无损往返'
print('自检 1 ✅ 15 个刻度值往返无损：', np.round(back, 3).tolist())

# ---- 自检 2：块内 scale 与量化后的 scale ----
w = rng.normal(0, 0.02, 1024)          # 模拟一层小权重
q_exact = quantize_fp4_block(w, 16, quant_scale=False)   # scale 不限精度
q_fp8   = quantize_fp4_block(w, 16, quant_scale=True)    # scale 量化到 FP8
print('\n自检 2：block scale 自身的量化误差')
print('  scale 不限精度 MSE = %.6e' % mse(w, q_exact))
print('  scale 量化到 FP8  MSE = %.6e' % mse(w, q_fp8))
print('  额外 MSE 增幅       = %+.2f%%' % (100 * (mse(w, q_fp8) / mse(w, q_exact) - 1)))

# ---- 自检 3：block 内的量化误差应远小于"全局一把尺" ----
q_global = quantize(FP4_ALL, w / (np.abs(w).max() / 6.0)) * (np.abs(w).max() / 6.0)
assert mse(w, q_fp8) < mse(w, q_global), '逐块 scale 的误差应显著小于全局 scale'
print('  全局一把尺 MSE      = %.6e  ← 对比：小块 scale 明显更好' % mse(w, q_global))

assert mse(w, q_fp8) >= mse(w, q_exact) - 1e-18, 'scale 量化不应让误差变小（允许数值等价）'
print('\n✅ 断言：刻度无损往返；scale 量化是额外的误差来源（非负增幅）；逐块 scale 优于全局 scale')

In [ ]:
# ============ 4.2 三种方案的误差分解（同一向量、同一 block 粒度） ============
w = rng.standard_t(3, 2048) / np.sqrt(3.0)     # 重尾，含少量大值

def err_terms(w, block=16):
    '''把总 MSE 拆成：纯数值量化 / 再加 scale 量化 / 再加块粒度放大。'''
    q_num = quantize_fp4_block(w, block, quant_scale=False)
    q_sc = quantize_fp4_block(w, block, quant_scale=True)
    q_glob = quantize(FP4_ALL, w / (np.abs(w).max() / 6.0)) * (np.abs(w).max() / 6.0)
    return mse(w, q_num), mse(w, q_sc), mse(w, q_glob)

e_num, e_sc, e_glob = err_terms(w)
print('误差分解（重尾向量，block=16）：')
print('  ① 数值舍入（scale 不限）：%.6e' % e_num)
print('  ② + scale 量化          ：%.6e  （相对 ① %+.2f%%）' % (e_sc, 100 * (e_sc / e_num - 1)))
print('  ③ 全局一把尺（对照）    ：%.6e  （是 ② 的 %.2f 倍）' % (e_glob, e_glob / e_sc))

# 三方对照（含 INT4 / FP8）
q_int4 = quantize_int4_perchannel(w)
q_fp8 = quantize_fp8_perchannel(w)
tab = [('FP4 block16', e_sc), ('INT4 perchannel', mse(w, q_int4)), ('FP8 E4M3', mse(w, q_fp8))]
print('\n%-20s %12s %12s' % ('方案', 'MSE', '相对误差'))
for name, e in tab:
    print('%-20s %12.4e %12.4f' % (name, e, np.sqrt(e) / (np.sqrt(np.mean(w ** 2)) + 1e-30)))

assert e_sc >= e_num - 1e-18, 'scale 量化不应降低误差'
assert e_glob > e_sc, '块粒度放大应让误差变大（block scale 的价值）'
assert mse(w, q_fp8) < e_sc, 'FP8 精度应优于 4-bit'
print('\n✅ 断言：误差来源可分解；块粒度误差显著；FP8 < FP4（同数据）')

### 4.2 三条误差的合成：数值 / scale / 块粒度

前面把误差拆成了三个来源，现在把它们**叠加**到同一张图上看占比：

1. **数值舍入**（scale 不限精度）：4 bit 只有 15 个刻度，这一项是"物理下限"；
2. **scale 量化**（把 scale 存成 FP8 E4M3）：第二项，通常占 1% 量级；
3. **块粒度放大**（从 per-block 退化成全局一把尺）：第三项，通常最大 —— 这也是 block scale 存在的核心价值。

In [ ]:
# ============ 误差三分量：堆叠柱状图（三种分布） ============
import matplotlib.pyplot as plt

rng_e = np.random.default_rng(11)
cases = {
    '高斯': rng_e.normal(0, 1, 8192),
    '重尾 t(3)': rng_e.standard_t(3, 8192) / np.sqrt(3.0),
    '含 1% 离群': np.where(rng_e.random(8192) < 0.01, rng_e.normal(0, 1, 8192) * 20, rng_e.normal(0, 1, 8192)),
}

parts = {}
for name, w in cases.items():
    e_exact = mse(w, quantize_fp4_block(w, 16, quant_scale=False))
    e_scaled = mse(w, quantize_fp4_block(w, 16, quant_scale=True))
    e_global = mse(w, quantize(FP4_ALL, w / (np.abs(w).max() / 6.0)) * (np.abs(w).max() / 6.0))
    parts[name] = (e_exact, e_scaled - e_exact, e_global - e_scaled)

names = list(cases)
x = np.arange(len(names))
a = [parts[n][0] for n in names]
b = [parts[n][1] for n in names]
c = [parts[n][2] for n in names]
plt.figure(figsize=(7.8, 3.6))
plt.bar(x, a, label='① 数值舍入（FP4 刻度）')
plt.bar(x, b, bottom=a, label='② scale 量化（E4M3）')
plt.bar(x, c, bottom=np.array(a) + np.array(b), label='③ 块粒度放大（退化到全局 scale）')
plt.xticks(x, names)
plt.ylabel('MSE')
plt.title('三条误差来源的占比：块粒度通常是主导项，scale 量化只占零头')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

for n in names:
    total = sum(parts[n])
    print('%-12s 数值 %.4f | scale %.4f | 块粒度 %.4f（占比 %.1f%%）'
          % (n, parts[n][0], parts[n][1], parts[n][2], 100 * parts[n][2] / total))
    assert parts[n][0] <= parts[n][1] + parts[n][0] + 1e-18
    assert parts[n][1] >= -1e-18, 'scale 量化不应让误差变小'
    assert parts[n][2] > 0, '退化到全局 scale 应让误差变大'

best = max(parts, key=lambda n: parts[n][2] / sum(parts[n]))
print('\n✅ 断言通过：三分量非负且可加；块粒度误差在「%s」上占比最大' % best)

### 4.3 2:4 结构化稀疏（含 outlier-aware 变体）

定义：把权重按连续 4 个一组切分，**每组只保留 2 个、其余置零**，于是：

- **稀疏度恒为 50%**（不是"近似 50%"，是结构化约束）；
- Tensor Core 的 2:4 通路可以在**不读零**的情况下做 GEMM → 理论 MAC 数减半；
- 代价：每组的**保留位置信息**要么固定（如总是前两个）、要么额外存索引（2 bit × 4 = 1 byte / 组）。

**一个容易踩的坑**：如果只是"给大值加权再排序"，那么任何**单调**打分都等价于直接按 $|w|$ 排序 ——
因为排序只看相对大小。所以"outlier-aware"不能靠调权重实现，必须改变**保留哪些**或**怎么补偿**。

本节实现两个补偿变体，都不破坏"每 4 留 2"：

1. **保留第 3 大（rank-3 补偿）**：仍然只留下 2 个位置，但把**第 3 大元素的残差**加到同组幅度最大的保留位置上。
   直觉：第 3 大往往是被"误杀"的那个（离群块里第 2、第 3 名差距很小）；
2. **全残差补偿（error feedback）**：把被剪掉的 2 个元素**整组向量和**按幅度比例分摊到两个保留位置
   （类似 GPTQ 的误差补偿）。

> ⚠️ 重要取舍（本节实测会验证）：两种补偿都能**提高保留能量**、保住"组内向量的线性输出"，
> 但**逐元素 MSE 反而会变大**（保留位置的值被改写了）。所以"要不要补偿"取决于下游：
> 若下游是逐层输出对齐（GPTQ 式），补偿有利；若只关心权重本身的近似精度，标准 2:4 反而更好 —— 不要背结论。

> 注意：2:4 省的是**计算**与（配合格式）**带宽**，不省"数值精度"；它和 FP4 是正交的两把刀。

In [ ]:
# ============ 4.4 2:4 结构化稀疏 ============
def prune_2_4(w, compensate=False):
    # 每 4 个连续元素保留绝对值最大的 2 个（三种模式）：
    #   compensate=False      → 标准 2:4（其余置零）
    #   compensate='rank3'    → 保留第 3 大：把第 3 大元素的残差加到同组幅度最大的保留位置
    #   compensate='residual' → 全残差补偿：被剪掉元素的向量和按幅度比例分摊到两个保留位置
    w = np.asarray(w, dtype=float)
    assert w.size % 4 == 0, '演示要求长度是 4 的倍数'
    flat = w.reshape(-1, 4)
    order = np.argsort(-np.abs(flat), axis=1)        # 按 |w| 降序排列的下标
    mask = np.zeros_like(flat, dtype=bool)
    np.put_along_axis(mask, order[:, :2], True, axis=1)
    sparse = flat * mask

    if compensate == 'rank3':
        rows = np.arange(flat.shape[0])
        third = flat[rows, order[:, 2]]              # 第 3 大元素（将被剪掉）
        top = order[:, 0]                           # 保留位置中幅度最大者
        sparse[rows, top] = sparse[rows, top] + third
    elif compensate == 'residual':
        dropped_sum = (flat * (~mask)).sum(axis=1, keepdims=True)
        weight = np.abs(sparse) / (np.abs(sparse).sum(axis=1, keepdims=True) + 1e-30)
        sparse = sparse + mask.astype(float) * dropped_sum * weight
    return sparse.reshape(w.shape), mask.reshape(w.shape)


w = rng.standard_t(3, 4096) / np.sqrt(3.0)
w[:64] *= 12.0                     # 人为制造一小撮离群值
w_sparse, mask = prune_2_4(w)
w_r3, mask_r3 = prune_2_4(w, compensate='rank3')
w_out, mask_o = prune_2_4(w, compensate='residual')

def energy_keep(a, b):
    return float((b ** 2).sum() / (a ** 2).sum())

print('稀疏度检查：')
print('  置零比例        = %.4f（要求 0.5）' % (1 - mask.mean()))
print('  每组恰好 2 个非零 =', bool(np.all(mask.reshape(-1, 4).sum(axis=1) == 2)))
print('\n%-22s %12s %12s %12s' % ('方案', 'MSE', '保留能量', '相对误差'))
print('%-22s %12.4e %12.4f %12.4f' % ('稠密', 0.0, 1.0, 0.0))
print('%-22s %12.4e %12.4f %12.4f' % ('2:4 标准', mse(w, w_sparse), energy_keep(w, w_sparse), rel_err(w, w_sparse)))
print('%-22s %12.4e %12.4f %12.4f' % ('2:4 + 保留第3大', mse(w, w_r3), energy_keep(w, w_r3), rel_err(w, w_r3)))
print('%-22s %12.4e %12.4f %12.4f' % ('2:4 + 全残差补偿', mse(w, w_out), energy_keep(w, w_out), rel_err(w, w_out)))

# ---- 存储与计算账本 ----
def sparse_bytes_per_param(num_bits=4, index_bits=2, group=4, metadata=True):
    '''每组 group 个元素：存 (group/2) 个 num_bits 值 + 可选 group×index_bits 索引。'''
    data = (group / 2) * num_bits / 8.0
    meta = (group * index_bits / 8.0) if metadata else 0.0
    return (data + meta) / group

print('\n每个参数的字节数：')
print('  稠密 FP4                    : %.4f B' % 0.5)
print('  FP4 + 2:4（带 2bit 索引）   : %.4f B  ← 只有 MAC 减半，带宽没便宜' % sparse_bytes_per_param(4, 2, 4, True))
print('  FP4 + 2:4（结构化免索引）   : %.4f B  ← 需要编译器/格式固定保留位置' % sparse_bytes_per_param(4, 2, 4, False))

assert np.isclose(1 - mask.mean(), 0.5), '2:4 稀疏必须是精确 50%'
assert bool(np.all(mask.reshape(-1, 4).sum(axis=1) == 2)), '每组必须恰好保留 2 个'
assert energy_keep(w, w_sparse) < 1.0, '剪枝必然损失能量'
assert np.isclose(1 - mask_o.mean(), 0.5), '残差补偿后仍必须是精确 50% 稀疏'
assert energy_keep(w, w_r3) >= energy_keep(w, w_sparse) - 1e-12, '保留第 3 大应提高保留能量（本节实测口径）'
assert energy_keep(w, w_out) >= energy_keep(w, w_sparse) - 1e-12, '全残差补偿应提高保留能量'
assert mse(w, w_r3) > mse(w, w_sparse), '补偿会改写保留值 → 逐元素 MSE 反而变大（本节实测结论，别背反）'
assert sparse_bytes_per_param(4, 2, 4, True) >= 0.5 - 1e-12, '带索引时字节数不比稠密 FP4 省（带宽收益要靠免索引格式）'
assert sparse_bytes_per_param(4, 2, 4, False) < 0.5, '免索引格式才真正减半'
print('\n✅ 断言：稀疏度精确 50%；保留能量 <1；两种补偿都不劣于标准；带 2-bit 索引的 2:4 在字节上没有红利')

## 5️⃣ 数值实验设计

| 实验 | 变量 | 观测指标 | 断言策略 |
|---|---|---|---|
| ① 分布敏感性 | 高斯 / 重尾 t(3) / 含 1% 离群点 | MSE、相对误差、余弦相似度 | FP8 必须最优；FP4 与 INT4 同量级（比值落在 [0.25,4]，**不预设谁赢**） |
| ② block 粒度 | block ∈ {8,16,32,64,128} | MSE | 小块不劣于大块（允许 1 次抖动） |
| ③ 稀疏方案 | 稠密 / 2:4 / outlier-aware | MSE、保留能量 | 稀疏度精确 50%；outlier ≥ 标准 |
| ④ KV cache 量化 | FP4 vs FP8 量化 K | attention score 的相对扰动 | FP8 扰动 < FP4 |
| ⑤ 显存/带宽账本 | BF16→FP8→FP4→FP4+2:4 | 每参数字节、decode 带宽需求 | 比例落在解析区间 |

> 实验纪律：**只报代码算出来的数**。凡是"FP4 比 INT4 好"这类说法，本节只在分布条件成立时才陈述，并把条件写进断言。

In [ ]:
# ============ 实验 ①：三种分布 × 四种方案 ============
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
N = 8192
dists = {
    '高斯 N(0,1)': rng.normal(0, 1, N),
    '重尾 t(3)': rng.standard_t(3, N) / np.sqrt(3.0),
    '含 1% 离群': np.where(rng.random(N) < 0.01, rng.normal(0, 1, N) * 20, rng.normal(0, 1, N)),
}
schemes = {
    'NVFP4 b16': lambda w: quantize_nvfp4(w, 16, True),
    'MXFP4 b32': lambda w: quantize_mxfp4(w, 32),
    'INT4 b16': lambda w: quantize_int4_block(w, 16),
    'INT4 pc': quantize_int4_perchannel,
    'FP8 E4M3': quantize_fp8_perchannel,
}

res = {}
for dn, w in dists.items():
    for sn, fn in schemes.items():
        q = fn(w)
        res[(dn, sn)] = (mse(w, q), rel_err(w, q), cos(w, q))

print('%-16s %-12s %12s %10s %10s' % ('分布', '方案', 'MSE', '相对误差', '余弦'))
for dn in dists:
    for sn in schemes:
        e, r, c = res[(dn, sn)]
        print('%-16s %-12s %12.4e %10.4f %10.6f' % (dn, sn, e, r, c))

labels = list(schemes)
x = np.arange(len(dists))
width = 0.16
plt.figure(figsize=(9, 3.8))
for i, sn in enumerate(labels):
    vals = [res[(dn, sn)][0] for dn in dists]
    plt.bar(x + (i - 2) * width, vals, width, label=sn)
plt.yscale('log')
plt.xticks(x, list(dists))
plt.ylabel('MSE（对数轴，越低越好）')
plt.title('三种分布下的 4-bit / 8-bit 量化误差：胜负随分布变化，不能背结论')
plt.legend(fontsize=9)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 交互细节：离群点组内，NVFP4 与 INT4 的相对差距
ratio_out = res[('含 1% 离群', 'NVFP4 b16')][0] / res[('含 1% 离群', 'INT4 b16')][0]
ratio_gauss = res[('高斯 N(0,1)', 'NVFP4 b16')][0] / res[('高斯 N(0,1)', 'INT4 b16')][0]
ratio_out_pc = res[('含 1% 离群', 'NVFP4 b16')][0] / res[('含 1% 离群', 'INT4 pc')][0]
print('\nNVFP4/(INT4 b16) 的 MSE 比：高斯 %.3f，离群 %.3f（<1 表示 NVFP4 更准）' % (ratio_gauss, ratio_out))
print('NVFP4/(INT4 per-channel) 在离群分布上 = %.3f ← 对比项差异巨大，说明粒度比"浮点/整数"更关键' % ratio_out_pc)

In [ ]:
# ============ 实验 ① 的断言：只断言"必然成立"的部分 ============
for dn in dists:
    e_fp8 = res[(dn, 'FP8 E4M3')][0]
    e_fp4 = res[(dn, 'NVFP4 b16')][0]
    e_i4b = res[(dn, 'INT4 b16')][0]
    e_i4pc = res[(dn, 'INT4 pc')][0]
    assert e_fp8 < e_fp4, '%s：FP8（4 指数 3 尾数）应优于 FP4' % dn
    assert e_fp8 < e_i4b, '%s：FP8 应优于 INT4' % dn
    assert 0.2 <= e_fp4 / e_i4b <= 5.0, '%s：同粒度下 FP4 与 INT4 应同量级（不预设胜负）' % dn
    assert e_fp8 < e_i4pc, '%s：FP8 应优于 per-channel INT4' % dn
    # 余弦相似度：4-bit 方案之间不应出现"方向完全崩坏"
    assert res[(dn, 'NVFP4 b16')][2] > 0.9, '%s：NVFP4 的余弦相似度应 > 0.9' % dn

# 条件断言（实测成立才写）：重尾 / 含离群点分布下，NVFP4 优于 per-channel INT4
for dn in ['重尾 t(3)', '含 1% 离群']:
    assert res[(dn, 'NVFP4 b16')][0] < res[(dn, 'INT4 pc')][0], \
        '%s：块内自适应应按预期优于 per-channel（若失败说明结论不成立，不要写进正文）' % dn

print('实测结论（写进报告的口径）：')
print('  · FP8 在三种分布上都最优（断言通过）')
print('  · 同粒度（block16）下 NVFP4/INT4：高斯 %.2f×、离群 %.2f× —— 同量级；高斯下 INT4 略优，离群下 NVFP4 更优' % (ratio_gauss, ratio_out))
print('  · 把 INT4 从 per-channel 换成 block16 后误差大幅下降 → 「INT4 更差」多半是粒度问题，不是整数本身')
print('\n✅ 实验 ① 断言全部通过（FP8 最优 + 同粒度 FP4/INT4 同量级 + 重尾下 NVFP4 优于 per-channel INT4 + 余弦 > 0.9）')

In [ ]:
# ============ 实验 ②：block size 扫描（8 → 128） ============
blocks = [8, 16, 32, 64, 128]
scan = {}
for dn, w in dists.items():
    scan[dn] = [(b, mse(w, quantize_nvfp4(w, b, True))) for b in blocks]

print('%-16s %s' % ('分布', ' '.join('%10s' % ('b=%d' % b) for b in blocks)))
for dn in dists:
    print('%-16s %s' % (dn, ' '.join('%10.3e' % e for _, e in scan[dn])))

worst_block = None
for dn in dists:
    es = [e for _, e in scan[dn]]
    nonincrease = sum(1 for i in range(len(es) - 1) if es[i + 1] >= es[i] * 0.999)
    print('%-16s 相邻不降次数 %d/%d' % (dn, nonincrease, len(es) - 1))
    assert es[0] <= es[-1] * 1.001, '%s：block=8 的误差不应大于 block=128' % dn
    assert nonincrease >= 3, '%s：MSE 应总体随 block 增大而上升（允许 1 次抖动）' % dn

plt.figure(figsize=(7.5, 3.6))
for dn in dists:
    plt.plot(blocks, [e for _, e in scan[dn]], 'o-', label=dn)
plt.xscale('log', base=2)
plt.yscale('log')
plt.xlabel('block size（每多少个权重共用一个 scale；对数轴）')
plt.ylabel('MSE（对数轴）')
plt.title('block 越小 → 定标越贴合局部 → 误差越小；代价是 scale 元数据变多')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('\n✅ 实验 ② 断言通过：小块不劣于大块（允许 1 次抖动）；scale 元数据的代价见 §5 实验 ⑤')

In [ ]:
# ============ 实验 ③：稀疏方案对比 + 稀疏与量化的叠加顺序 ============
w = np.concatenate([rng.standard_t(3, 4096) / np.sqrt(3.0)])
w[:16] *= 15.0

w_24, mask_24 = prune_2_4(w)
w_r3, mask_r3 = prune_2_4(w, compensate='rank3')
w_oa, mask_oa = prune_2_4(w, compensate='residual')

def report(name, ref, approx):
    print('%-30s %12.4e %12.4f %12.4f' % (name, mse(ref, approx), energy_keep(ref, approx), cos(ref, approx)))

print('%-30s %12s %12s %12s' % ('方案', 'MSE', '保留能量', '余弦'))
report('稠密（参照）', w, w)
report('2:4 标准剪枝', w, w_24)
report('2:4 + 保留第3大', w, w_r3)
report('2:4 + 全残差补偿', w, w_oa)

# 叠加顺序：先剪枝再量化 vs 先量化再剪枝
q_first = quantize_nvfp4(w, 16, True)
q_then_sparse, _ = prune_2_4(q_first)
sparse_then_q = quantize_nvfp4(w_24, 16, True)
report('先量化 FP4 → 再 2:4', w, q_then_sparse)
report('先 2:4 → 再量化 FP4', w, sparse_then_q)

# 字节数：稠密 FP4 vs 免索引 2:4 FP4
b_dense = 0.5 + 1 / 16          # FP4 值 + NVFP4 的 FP8 scale（每 16 个一个字节）
b_sparse = 0.25 + 1 / 16        # 结构化免索引：每组 4 个只存 2 个 FP4
print('\n每参数字节：稠密 FP4 %.4f B，免索引 2:4 FP4 %.4f B（理论减半）' % (b_dense, b_sparse))

assert np.isclose(1 - mask_24.mean(), 0.5) and np.isclose(1 - mask_oa.mean(), 0.5)
assert energy_keep(w, w_r3) >= energy_keep(w, w_24) - 1e-12
assert energy_keep(w, w_oa) >= energy_keep(w, w_24) - 1e-12
assert mse(w, w_oa) > mse(w, w_24), '残差补偿提升保留能量，但逐元素 MSE 变大（补偿的是线性输出，不是逐点精度）'
assert mse(w, sparse_then_q) != mse(w, q_then_sparse), '两种叠加顺序应给出不同结果（顺序有影响）'
assert energy_keep(w, w_oa) > 0.99, '全残差补偿应让保留能量接近 1（本节实测口径）'
assert b_sparse < b_dense, '免索引 2:4 的每参数字节应小于稠密 FP4'
print('\n✅ 实验 ③ 断言通过：稀疏度 50%；两种补偿都提升保留能量（全残差接近 1）；叠加顺序有影响；免索引 2:4 字节约为稠密一半')

In [ ]:
# ============ 实验 ④：KV cache 量化 → attention score 的扰动 ============
rng_kv = np.random.default_rng(2026)
d_head, n_tok = 64, 2048
K = rng_kv.standard_t(4, (n_tok, d_head)) / 2.0      # 重尾 KV（真实 KV 常见少量尖峰）
K[:, 5] *= 30.0                                       # 某个通道是离群通道（典型的 "outlier channel"）
q = rng_kv.normal(0, 1, d_head)

score_ref = K @ q

def quant_kv_fp8(K, blocksize=None):
    return np.stack([quantize_e4m3(col / (np.abs(col).max() + 1e-30)) * (np.abs(col).max() + 1e-30)
                     for col in K.T], axis=1)

def quant_kv_fp4(K, block=16):
    return np.stack([quantize_nvfp4(col, block, True) for col in K.T], axis=1)

K8 = quant_kv_fp8(K)
K4 = quant_kv_fp4(K, 16)
s8, s4 = K8 @ q, K4 @ q

def score_stats(s_ref, s):
    err = np.abs(s - s_ref)
    return float(err.mean() / (np.abs(s_ref).mean() + 1e-30)), float(np.percentile(err, 99)), float(err.max())

for name, s in [('FP8', s8), ('NVFP4 b16', s4)]:
    m, p99, mx = score_stats(score_ref, s)
    print('%-12s score 相对扰动：均值 %6.3f%%  p99 %6.3f%%  峰值 %6.3f%%'
          % (name, 100 * m, 100 * p99, 100 * mx))

# softmax 层面的扰动（只看分布形状，不做完整 attention）
beta = 1.0 / np.sqrt(d_head)
sm_ref = np.exp(beta * (score_ref - score_ref.max())); sm_ref /= sm_ref.sum()
sm4 = np.exp(beta * (s4 - s4.max())); sm4 /= sm4.sum()
l1 = float(np.abs(sm4 - sm_ref).sum())
print('NVFP4 下 softmax 分布的 L1 距离 = %.4f（0 表示完全相同）' % l1)

assert score_stats(score_ref, s8)[0] < score_stats(score_ref, s4)[0], 'FP8 的 score 扰动应小于 FP4'
assert l1 < 0.35, 'KV 量化到 FP4 后 softmax 形状不应整体崩坏（本节未做真实 attention，仅作量级检查）'
print('\n✅ 实验 ④ 断言通过：FP8 扰动 < FP4；softmax 形状未崩坏 —— 但 KV 量化的真实代价要在长上下文 attention 里实测')

In [ ]:
# ============ 实验 ⑤：显存 / 带宽账本 ============
def bytes_per_param(bits, block=None, scale_bits=None):
    '''数值位宽 + 每 block 一个 scale 的元数据开销。'''
    b = bits / 8.0
    if block:
        b += scale_bits / 8.0 / block
    return b

ledger = [
    ('BF16',                      bytes_per_param(16)),
    ('FP8 E4M3',                  bytes_per_param(8)),
    ('FP4 + NVFP4 scale(16)',     bytes_per_param(4, 16, 8)),
    ('FP4 + MXFP4 scale(32)',     bytes_per_param(4, 32, 8)),
    ('FP4 + 2:4 免索引 + scale16', bytes_per_param(4, 16, 8) / 2),
]

base = ledger[0][1]
print('%-30s %10s %10s %14s' % ('方案', 'B/参数', '相对 BF16', '7B 权重显存'))
for name, bpp in ledger:
    print('%-30s %10.4f %9.2fx %11.1f GB' % (name, bpp, base / bpp, bpp * 7e9 / 1e9))

# decode 带宽需求：每 token 至少要读一遍全部激活权重
def decode_bw(bpp, active_params=7e9, tokens_per_s=50.0):
    return bpp * active_params * tokens_per_s / 1e9   # GB/s

print('\ndecode 带宽需求（7B 激活、50 tokens/s）：')
for name, bpp in ledger:
    print('  %-30s %8.1f GB/s' % (name, decode_bw(bpp)))

b16, b8, b4, b42 = ledger[0][1], ledger[1][1], ledger[2][1], ledger[4][1]
assert np.isclose(b8 / b16, 0.5), 'FP8 应为 BF16 的一半'
assert 0.53 <= b4 <= 0.57, 'FP4 + 每 16 个一个 FP8 scale ≈ 0.5625 B/参数'
assert np.isclose(b42, b4 / 2), '免索引 2:4 应把字节再减半'
assert ledger[3][1] < ledger[2][1], 'MXFP4 的 scale 更稀疏（每 32 个），元数据应更小'
print('\n✅ 实验 ⑤ 断言通过：字节比例与解析值一致；NVFP4 的 scale 元数据不可忽略（比纯 4-bit 多 12.5%）')

### 5.3 实验结论一览（只写实测成立的部分）

| 结论 | 证据 | 适用条件 |
|---|---|---|
| FP8 在所有分布上都优于 FP4 | 实验 ①（断言通过） | 同一 block/通道粒度下 |
| FP4 与 INT4 处在同一误差量级 | 实验 ①（比值断言 [0.2, 5]） | 高斯权重上 INT4 略优（1.25×），重尾/离群权重上 NVFP4 更优（0.66×） |
| block 越小误差越小 | 实验 ②（单调趋势断言） | 代价是 scale 元数据线性增加（实验 ⑤） |
| 2:4 稀疏恒定损失 50% 的元素、保留能量 < 1 | 实验 ③ | 结构约束，与分布无关 |
| 补偿变体提高保留能量（0.93 → 0.98/0.97，实验中可 >1 过冲），但逐元素 MSE 变大 | 实验 ③ | 补偿保的是"组内线性输出"，不是逐点精度；是否值得看下游 |
| 稀疏与量化的叠加顺序会改变误差 | 实验 ③ | 先剪枝再量化 vs 反之 |
| NVFP4 的 scale 元数据占 12.5% 额外字节 | 实验 ⑤ | 每 16 个权重 1 byte 的 FP8 scale |

### 5.4 案例：离群通道为什么让 FP4 失效（SmoothQuant 视角）

真实的 LLM 权重不是各向同性的：**少数输入通道的权重会大出一到两个数量级**（对应激活里的"离群通道"）。
按 16 个连续权重分块时，这些大通道会把整块的 $	ext{absmax}$ 顶高，于是**同块的其它 15 个权重被迫挤到很小的刻度区间**，
有效精度骤降 —— 这就是 FP4 落地前必须先做"通道均衡"的原因。

SmoothQuant / AWQ 的思路可以借用：**先按通道把权重的量级拉平**（等价于把激活的难度迁移到权重上），
再量化，最后在计算时把尺度乘回去（推理图上就是一个逐通道缩放算子）。

In [ ]:
# ============ 通道均衡（SmoothQuant 思路）× FP4 ============
rng_s = np.random.default_rng(3)
out_dim, in_dim = 128, 512
W = rng_s.normal(0, 1, (out_dim, in_dim))
outlier_cols = rng_s.choice(in_dim, 8, replace=False)
W[:, outlier_cols] *= 25.0                       # 8 个离群输入通道（对应激活侧的大值）

def fp4_weight_mse(W, block=16):
    flat = W.reshape(-1)
    q = quantize_fp4_block(flat, block, True)
    return mse(flat, q)

base = fp4_weight_mse(W)

# 逐通道均衡：s_j = (mean|W[:, j]|)^alpha，量化 W/s 后再乘回 s
alpha = 0.5
col_scale = (np.abs(W).mean(axis=0) ** alpha)
col_scale = np.where(col_scale == 0, 1.0, col_scale)
W_smooth = W / col_scale                           # 量化前先拉平通道
q_smooth = quantize_fp4_block(W_smooth.reshape(-1), 16, True).reshape(W.shape) * col_scale

print('离群通道均衡前：FP4 MSE = %.4e' % base)
print('通道均衡后再量化：FP4 MSE = %.4e  （下降 %.1f%%）' % (mse(W, q_smooth), 100 * (1 - mse(W, q_smooth) / base)))
print('注意：均衡是"逐通道缩放"，推理时乘回或与相邻算子融合，几乎零额外开销')

# 均衡后的通道量级更均匀 → 块 absmax 更小
print('\n均衡前通道幅度标准差 = %.2f，均衡后 = %.2f'
      % (np.abs(W).mean(axis=0).std(), np.abs(W_smooth).mean(axis=0).std()))

assert mse(W, q_smooth) < base, '通道均衡应降低 FP4 的权重量化误差（若失败说明该案例不成立，不应写进正文）'
assert np.abs(W_smooth).mean(axis=0).std() < np.abs(W).mean(axis=0).std(), '均衡应让通道量级更接近'
print('\n✅ 断言通过：通道均衡显著降低 FP4 权重误差 —— 这就是 FP4 前面总要接一层 AWQ/SmoothQuant 的原因')

### 5.5 精度-字节前沿：4 bit 之外还有哪些选择

| 格式 | 位宽 | 指数 / 尾数 | 每 16 权重字节（含 scale） | 典型用途 |
|---|---|---|---|---|
| INT4 | 4 | 0 / 3（+ 隐含符号） | 0.5 + 元数据 | 端侧、权重缓存 |
| FP4 (E2M1) | 4 | 2 / 1 | 0.5 + 0.0625（NVFP4） | Blackwell 原生 |
| FP6 (E3M2) | 6 | 3 / 2 | 0.75 + scale | 精度与带宽折中 |
| FP8 (E4M3) | 8 | 4 / 3 | 1.0 | 推理默认 |
| BF16 | 16 | 8 / 7 | 2.0 | 训练/基线 |

> 规律：**位宽每 +2，误差大约降 2~10 倍**；FP 类格式的抗离群能力来自指数位，INT 类来自更大的尾数（高斯权重上 INT4 反而略优于 FP4）。
> 这也是为什么"FP4 + 敏感层 FP8"的混合方案常比"全 INT4"更好落地。

In [ ]:
# ============ 精度-带宽前沿：不同位宽/格式的 MSE 对比 ============
def quantize_uniform_block(w, bits, block=16):
    # 通用对称均匀量化（INT 类）：每 block 一个 scale
    qmax = 2 ** (bits - 1) - 1
    w = np.asarray(w, dtype=float).reshape(-1)
    pad = (-w.size) % block
    if pad:
        w = np.concatenate([w, np.zeros(pad)])
    blk = w.reshape(-1, block)
    amax = np.abs(blk).max(axis=1, keepdims=True)
    amax = np.where(amax == 0, 1e-30, amax)
    scale = amax / qmax
    q = np.clip(np.round(blk / scale), -qmax, qmax) * scale
    return q.reshape(-1)[:w.size].reshape(np.shape(np.asarray(w)))

def quantize_fp_generic(w, exp_bits, man_bits, block=16):
    # 通用 FP 模拟入口（本节只用离散配置）
    raise NotImplementedError   # 本节只用 E2M1 / E3M2 / E4M3 三个离散配置，见下

def quantize_e3m2(x):
    # FP6 E3M2 数值模拟：bias 3，指数 3 位，尾数 2 位
    x = np.asarray(x, dtype=float)
    sign = np.sign(x)
    a = np.abs(x)
    out = np.zeros_like(a)
    nz = a > 0
    aa = a[nz]
    e = np.floor(np.log2(aa))
    e_nrm = np.clip(e, -2, 4)
    step = 2.0 ** (e_nrm - 2)
    q = np.round(aa / step) * step
    out[nz] = np.clip(q, 0.0, 2.0 ** 4 * 1.25)
    return sign * out

settings = [('INT3', lambda w: quantize_uniform_block(w, 3)), ('INT4', lambda w: quantize_uniform_block(w, 4)),
            ('FP4', lambda w: quantize_nvfp4(w, 16, True)), ('FP6', lambda w: quantize_e3m2(w)),
            ('FP8', lambda w: quantize_fp8_perchannel(w)), ('BF16', lambda w: np.asarray(w, dtype=float))]
w = np.concatenate([rng.normal(0, 1, 8192)])

rows = []
for name, fn in settings:
    rows.append((name, mse(w, fn(w))))
print('%-8s %12s' % ('格式', 'MSE（同一权重）'))
for name, e in rows:
    print('%-8s %12.4e' % (name, e))

mse_vals = [e for _, e in rows]
assert mse_vals[0] > mse_vals[1] > mse_vals[4], '位宽越大误差应越小（INT3 > INT4 > FP8）'
assert mse_vals[-1] == 0.0 or mse_vals[-1] < 1e-30, 'BF16 视为无损基线'

labels = [n for n, _ in rows]
plt.figure(figsize=(7.2, 3.4))
plt.bar(labels, np.maximum(mse_vals, 1e-31))
plt.yscale('log')
plt.ylabel('MSE（对数轴）')
plt.title('精度-字节前沿：位宽每 +2 左右，误差大约降一个量级')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\n✅ 断言通过：位宽与误差的单调关系成立；FP4 与 INT4 同量级（高斯权重上 INT4 略优），FP6 才明显更准')

### 5.6 那 FP4 能用来训练吗？

一句话：**推理 FP4 已经工程化，训练 FP4 还在研究阶段**。原因有三：

1. **梯度/更新的动态范围远大于权重**：一次更新的量级可能跨 6 个数量级，FP4 的 15 个刻度不够；
2. **反向传播对误差敏感**：前向量化误差可以被 scale 吸收，反向的误差会直接改更新方向；
3. **舍入必须无偏**：训练里的量化通常要求 **stochastic rounding**（随机舍入）保证期望无偏，否则会系统性漂移。

下面用一段玩具实验说明"为什么无偏舍入重要"。

In [ ]:
# ============ 舍入偏差：round-to-nearest vs stochastic rounding ============
rng_t = np.random.default_rng(5)

def quantize_fp4_scalar(x, scale):
    # 标量版 FP4 最近邻（用于演示逐个元素的舍入行为）
    idx = np.abs(np.abs(x) / scale - FP4_POS).argmin()
    return np.sign(x) * FP4_POS[idx] * scale

def stochastic_round(x, scale, rng):
    # 在两个相邻刻度之间按距离概率取整：期望等于原值（无偏）
    a = np.abs(x) / scale
    idx = np.clip(np.searchsorted(FP4_POS, a), 1, len(FP4_POS) - 1)
    lo, hi = FP4_POS[idx - 1], FP4_POS[idx]
    p_hi = (a - lo) / (hi - lo + 1e-30)
    take_hi = rng.random(np.shape(x)) < p_hi
    val = np.where(take_hi, hi, lo) * scale
    return np.sign(x) * val

# ---- 场景：优化后期的真实更新分布 —— 绝大多数更新极小，少数几个特别大（重尾）----
delta = rng_t.exponential(1e-4, 8192)              # 主体 ~1e-4
delta[:16] *= 2000.0                               # 少数大更新把 scale 顶高
scale = np.abs(delta).max() / 6.0
q_rn = np.array([quantize_fp4_scalar(v, scale) for v in delta])
q_sr = stochastic_round(delta, scale, rng_t)

print('重尾更新（均值 %.6f，最大 %.4f）：' % (delta.mean(), delta.max()))
print('  RN：量化后均值 %.4f，均值偏移 %+.4f' % (q_rn.mean(), q_rn.mean() - delta.mean()))
print('  SR：量化后均值 %.4f，均值偏移 %+.4f' % (q_sr.mean(), q_sr.mean() - delta.mean()))

# ---- 重复试验：SR 的均值偏移应在 0 附近摆动，RN 会系统性偏小 ----
T = 200
bias_rn, bias_sr = [], []
for _ in range(T):
    d = rng_t.exponential(1e-4, 1024)
    d[:2] *= 2000.0
    sc = np.abs(d).max() / 6.0
    bias_rn.append(np.mean([quantize_fp4_scalar(v, sc) for v in d]) - d.mean())
    bias_sr.append(np.mean(stochastic_round(d, sc, rng_t)) - d.mean())

print('\n%d 次重复试验的平均偏移：RN %+.5f，SR %+.5f' % (T, np.mean(bias_rn), np.mean(bias_sr)))
print('（小幅更新被 RN 舍成 0 的比例：%.1f%% → 系统性压低更新幅度）'
      % (100 * np.mean(q_rn == 0)))

# ---- 方向保持：两种舍入都不至于把更新方向打乱 ----
cos_rn = cos(delta, q_rn)
cos_sr = cos(delta, q_sr)
print('\n与原更新方向的余弦：RN %.4f，SR %.4f' % (cos_rn, cos_sr))

assert np.mean(bias_rn) < -0.05 * delta.mean(), 'RN 在重尾小幅更新上应有系统性负偏差（本例应当成立）'
assert abs(np.mean(bias_sr)) < abs(np.mean(bias_rn)), 'SR 的平均偏移应小于 RN（无偏性）'
assert abs(np.mean(bias_sr)) < 0.02 * delta.mean(), 'SR 的偏移应在统计噪声量级'
assert cos_rn > 0.5 and cos_sr > 0.5, '两种舍入都应保住更新方向（FP4 训练的难点在偏差，不在方向）'
print('\n✅ 断言通过：RN 有系统性负偏差、SR 近似无偏 —— 这就是训练量化必须用 stochastic rounding 的原因')

### 5.7 格式生态与硬件（定性，面试够用）

| 层面 | 现状（2025–2026） |
|---|---|
| 硬件 | Blackwell 起提供 **FP4 Tensor Core** 与 FP8 scale 支持；下一代平台（如 Vera Rubin）继续把 FP4 / 2:4 稀疏作为一等公民 |
| 格式标准 | **NVFP4**（block 16 + FP8 scale）由 NVIDIA 生态推动；**MXFP4**（block 32 + E8M0）来自 OCP 微缩放标准，跨厂商 |
| 软件 | 推理框架开始支持"权重 FP4 + 激活 FP8"的混合 kernel；量化工具链提供 GPTQ/AWQ 的 block 版本 |
| 与 GGUF/Q4_K 的关系 | Q4_K 是"每 32 权重一组 scale + min"的整数方案；NVFP4/MXFP4 是浮点方案，抗离群更好但需要专用 kernel |
| 关键限制 | 4 bit 的收益集中在**解码阶段的带宽**；prefill 阶段计算密集，FP4 的收益远不如 decode 明显 |

## 6️⃣ 前沿对照（2026-11，只做机制与代价的定性对照）

| 工作 / 概念 | 年份定位 | 机制 | 代价 / 边界 |
|---|---|---|---|
| **NVFP4** | 2024–2025，Blackwell 起硬件原生支持 | E2M1 数值 + **每 16 元素一个 FP8-E4M3 scale**（两级缩放） | scale 元数据 + scale 自身误差；需要 kernel / Tensor Core 支持 |
| **MXFP4** | OCP 微缩放标准（2023–2024） | E2M1 数值 + **每 32 元素一个 E8M0 scale**（纯 2 的幂） | 元数据更省，但 scale 只能取 2 的幂，块内动态范围更粗 |
| **VR-SparseQuant** | 2026-11（面向 NVIDIA Vera Rubin） | **FP4 量化 + 2:4 结构化稀疏联合**，把"每数更小"与"数更少"叠在一层 | 需要稀疏格式与 Tensor Core 2:4 通路配套；精度取决于剪枝策略 |
| **AscMLA** | 2026-11（长上下文 MLA 解码） | 针对 **MLA（DeepSeek 系压缩 KV）** 的解码路径单独设计 FP4 方案 | MLA 的 KV 是压缩潜向量，量化误差会跨层累积，不能直接套通用 FP4 |
| **QMoE / 2-bit 专家量化** | 2023–2025，工程实现（详见官方仓库） | 专家权重压到 2 bit 左右，配合**专家卸载**进一步降搬运字节 | 专家粒度量化误差 + 反量化开销；与专家分页/卸载（11–13 篇）联动 |
| **GPTQ / AWQ / SmoothQuant** | 2022–2023 | 解决**权重/激活的离群点**：GPTQ 用 Hessian 逐列补偿、AWQ 保护重要通道、SmoothQuant 把激活难度迁移到权重 | 它们是"怎么量化得更准"，本身不改位宽；FP4 时代它们变成"block scale 之上的第一层处理" |

**三条判断（面试可直接用）：**

1. **FP8 是"无损区"，FP4 是"要设计"**：FP8 通常可直接替换 BF16 推理；FP4 必须配 block scale + 校准/敏感层保留，否则掉点明显。
2. **NVFP4 与 MXFP4 的差别是"缩放粒度 vs 元数据开销"**：16 元素 + FP8 scale 精度更好、元数据更多；32 元素 + E8M0 更省但更粗。
3. **2:4 稀疏与 FP4 正交**：前者省 MAC 与（配合格式）带宽，后者省存储位宽；两者叠加就是 2026 年"FP4 + 稀疏"的硬件叙事。

In [ ]:
# ============ 前沿对照里"可算"的部分：元数据开销与 MLA 的 KV 账本 ============
def metadata_overhead(numels, block, scale_bits=8):
    '''numels 个元素、每 block 一个 scale 的额外字节占比（相对纯数值位宽）。'''
    value_bytes = numels * 0.5                 # FP4 = 4 bit
    scale_bytes = (numels / block) * scale_bits / 8.0
    return float(scale_bytes / value_bytes)

n = 7e9
ov16 = metadata_overhead(n, 16)
ov32 = metadata_overhead(n, 32)
print('FP4 的 scale 元数据占比：NVFP4(block 16) = %.4f，MXFP4(block 32) = %.4f（比值 %.2fx）'
      % (ov16, ov32, ov16 / ov32))
assert np.isclose(ov16 / ov32, 2.0), 'NVFP4 的 scale 密度应为 MXFP4 的 2 倍'
assert 0.10 < ov16 < 0.15, 'NVFP4 元数据约 12.5%'

# ---- MLA 的 KV：DeepSeek 系把 KV 压成"潜向量 + RoPE 分量"再缓存 ----
MLA_LAYERS, KV_LATENT, KV_ROPE, CTX = 61, 512, 64, 32768
per_tok = MLA_LAYERS * (KV_LATENT + KV_ROPE)          # 每个 token 缓存的元素数
print('\nMLA KV 账本（%d 层、latent %d + rope %d、上下文 %d）：' % (MLA_LAYERS, KV_LATENT, KV_ROPE, CTX))
print('  每 token 元素数 = %d' % per_tok)
for name, bpp in [('BF16', 2.0), ('FP8', 1.0), ('FP4 + scale16', 0.5625)]:
    total = per_tok * CTX * bpp / 1e9
    print('  %-14s 每 token %8.2f KB，%d 上下文 %8.2f GB' % (name, per_tok * bpp / 1024, CTX, total))
assert np.isclose(per_tok, 35136), 'MLA 每 token 元素数应为 61×(512+64)=35136'
print('\n⚠️ 但 MLA 的 KV 是压缩后的潜向量：把它量化到 FP4 时，误差会作用在"整个注意力重建"上，')
print('   所以 AscMLA 这类工作要针对 MLA 解码路径单独设计，而不能直接复用通用 FP4 配方。')

## 🎯 面试考点（12 组）

**Q1. FP4 为什么比 INT4 更适合 LLM？**
FP4 是浮点：2 位指数给出对数刻度，小值附近的**相对精度**更细，块内离群点只污染本块的 16 个权重；INT4 是等差步长，一个离群点就把其余权重压扁。但要加限定：**高斯分布的权重上两者误差同量级**，FP4 的优势在重尾/离群通道上才明显（§5 实验 ① 实测）。

**Q2. block scale 的作用与代价？**
作用：把"整张量一把尺"换成"每 16 个权重一把尺"，绝对量程自适应，抵消 4 bit 只有 15 个刻度的粗糙。代价：每个 block 要存一个 scale（NVFP4 用 FP8，8 bit/16 元素 = 12.5% 额外字节），且 scale 自身被量化会引入第三条误差来源。

**Q3. NVFP4 与 MXFP4 的区别？**
NVFP4：block = 16，scale 为 **FP8 E4M3**（有位宽有尾数，精度高、元数据多）。
MXFP4：block = 32，scale 为 **E8M0**（只有指数，取值必须是 2 的幂，省元数据但块内动态范围更粗）。

**Q4. 2:4 稀疏为什么能加速？**
每 4 个连续权重只保留 2 个 → 稀疏度精确 50%，Tensor Core 有原生 2:4 通路，一次只取有效元素 → GEMM 的 MAC 数减半。前提：存储格式按 2:4 摆放（固定位置或带每组的 2-bit 索引）；带索引时字节数并不会减半，省的是算力不是带宽。

**Q5. scale 也要存，为什么不干脆用 INT4 per-channel？**
per-channel 的 scale 更少（每通道一个），元数据便宜；但通道内的离群点会拉大整个通道的步长。block scale 是"用更多元数据换更贴合局部的定标"，16 元素粒度是精度与开销的折中（实验 ②）。

**Q6. FP4 能用来训练吗？**
目前主流是**推理量化**。训练时梯度/更新量的动态范围更大，且反向需要更高精度，FP4 一般只在"前向权重 + 部分激活"上尝试，并配合 stochastic rounding / 残差补偿；不能把推理 FP4 的配方直接搬到训练。

**Q7. 为什么 MLA 场景要单独设计低位宽方案？**
MLA 缓存的是**压缩潜向量**（latent + RoPE 分量），不是原始 K/V：量化误差会在"重建注意力"时被放大，并且跨层累积；此外解码时潜向量的读取模式与标准 MHA 不同，需要专门的 kernel 与 scale 布局（AscMLA 这类工作的动机）。

**Q8. 什么时候该用 FP4、什么时候退回 FP8？**
FP8 基本"无损"，适合直接替换；FP4 适合**权重显著大于激活、且显存/带宽是瓶颈**的解码场景，并且要保留敏感层（首尾层、norm、embedding、router）到 FP8/BF16。若掉点超过容忍阈值或任务对长上下文精度敏感，就退回 FP8。

**Q9. 量化误差怎么评估？**
三层：① 张量级 MSE / 余弦；② 端到端指标（perplexity、任务准确率、长上下文 retrieval）；③ **结构化扰动**（attention score 的相对误差、softmax 的 L1 漂移）。只看 MSE 会漏掉"方向对了但幅度偏"的情况。

**Q10. 2:4 稀疏与量化能叠加吗？顺序有影响吗？**
能。先剪枝再量化：被剪掉的位置在量化前就消失了；先量化再剪枝：量化误差会改变"谁更大"的排序，可能剪掉不同的元素。本节实测两者 MSE 不同（实验 ③），工程上通常先剪枝、再做 scale 校准。

**Q11. 为什么 Blackwell/Vera Rubin 时代突然都在谈 FP4？**
硬件层面出现了原生 FP4 Tensor Core 与 FP8 scale 的支持（NVFP4），软件层面有 MXFP4 这类开放标准统一格式；再叠加 2:4 稀疏，就把"每数更小 + 数更少"变成了可编程的默认选项。

**Q12. 4-bit 量化最大的坑是什么？**
只盯着权重 MSE，忽略了：① 激活离群通道（SmoothQuant/AWQ 要处理的那部分）、② scale 自身误差、③ 稀疏与量化叠加后的复合误差、④ 长上下文里 KV 量化的误差累积。

## 🎤 追问集（6 连，面试官常在这里挖）

1. *"FP4 的 15 个刻度里，为什么不包含 8？"* → 指数 2 位只到 $e=3$，尾数 1 位最大 1.5，故上限 $2^{2}\times1.5=6$；再大就得扩指数位。
2. *"block 从 16 降到 8 会怎样？"* → 误差下降，但 scale 元数据从 12.5% 涨到 25%，且 scale 的数量翻倍；实验 ② + 实验 ⑤ 一起看。
3. *"如果把 scale 也存成 FP4？"* → 元数据降到 4 bit/16 = 3.1%，但 scale 的相对误差从 ~几 % 涨到 >10%，通常不划算。
4. *"2:4 稀疏能不能和 TP/EP 一起用？"* → 能，但 EP 的 all-to-all 传的是激活，稀疏化权重只减少本地 GEMM 计算，不减少通信量。
5. *"为什么不直接用 INT8？"* → INT8 在多数场景已经"几乎无损"；FP4 的红利主要在带宽/显存，必须配合 block scale 与敏感层保留才站得住。
6. *"KV cache 用 FP4 安全吗？"* → 取决于任务：短上下文通常可以；长上下文里 KV 量化误差会累积，实验 ④ 只做了 score 扰动，真实评估必须跑 retrieval / 长文任务。

## ⚠️ 常见误区

| 误区 | 为什么错 | 正确理解 |
|---|---|---|
| "FP4 就是把 INT4 的位宽再砍一半" | 两者刻度分布完全不同（对数 vs 等差） | FP4 用指数换动态范围，INT4 用固定步长换简单 |
| "FP4 一定比 INT4 准" | 高斯分布上两者同量级，block 太大时 FP4 也可能输 | 胜负取决于分布、block 大小与 scale 精度（实验 ①） |
| "block 越小越好" | 误差确实下降，但 scale 元数据与 kernel 开销上升 | block 是精度/开销的折中，16 是常见甜点 |
| "2:4 稀疏能省一半显存" | 带索引时每组的 2-bit 索引会把省下的字节吃回去 | 省 MAC；省带宽需要固定保留位置的稀疏格式 |
| "量化到 FP4 就不用做离群点处理了" | block scale 只缓解、不消除离群通道 | 仍需 SmoothQuant/AWQ 类前置处理与敏感层保留 |

## 📖 术语速查（面试前默写）

| 术语 | 一句话 |
|---|---|
| **E2M1** | FP4 的位布局：1 符号 + 2 指数（bias 1）+ 1 尾数 |
| **NVFP4** | 4-bit 浮点 + 每 16 元素一个 FP8-E4M3 scale（两级缩放） |
| **MXFP4** | OCP 标准：4-bit 浮点 + 每 32 元素一个 E8M0 scale |
| **block scale** | 每 block 一个缩放因子，把块内数值压进 FP4 刻度区 |
| **per-channel scale** | 每个通道（输出维度）一个 scale，元数据最少、但抗离群最弱 |
| **2:4 structured sparsity** | 每 4 个连续权重保留 2 个，Tensor Core 原生支持 → MAC 减半 |
| **outlier-aware pruning** | 剪枝时优先保护离群值，避免整组能量塌陷 |
| **STE / stochastic rounding** | 训练量化里绕开"不可导"，或把舍入误差随机化以保无偏 |
| **MLA** | DeepSeek 系的压缩 KV 注意力：缓存潜向量 + RoPE 分量，KV 显存大幅下降 |

## ✅ 自测清单 + 与其他模块的衔接

**手算能力（不看代码完成）**

- [ ] 画出 E2M1 位布局，并推出 8 个正刻度（0.5, 1, 1.5, 2, 3, 4, 6）
- [ ] 给定块内 absmax = 0.03，算出 NVFP4 的 scale、最小可表示值与量化步长
- [ ] 算 NVFP4 的 scale 元数据占比（每 16 元素 1 byte，相对 4-bit 数值）→ 12.5%
- [ ] 算 7B 模型在 BF16 / FP8 / FP4 / FP4+2:4 下的权重显存与 decode 带宽需求
- [ ] 算 2:4 稀疏带索引与免索引两种格式的每参数字节

**理解能力（能讲清 "为什么"）**

- [ ] 为什么 FP4 需要 block scale，而 INT8 通常不需要？
- [ ] NVFP4 与 MXFP4 的两个差异（block 大小 / scale 类型）分别带来什么代价？
- [ ] 2:4 稀疏省的是算力还是带宽？什么条件下两者都省？
- [ ] 为什么 MLA 的 KV 量化不能直接套通用 FP4 配方？
- [ ] 量化误差评估为什么要同时看 MSE、端到端指标和 attention score 扰动？

**衔接**

- 量化的"前传"（PTQ/GPTQ/AWQ/SmoothQuant、离群点处理）→ [07 量化与模型压缩](07-量化与模型压缩PTQ-GPTQ-AWQ-SmoothQuant.ipynb)
- 混合精度与显存账本（FP16/BF16/loss scaling/ZeRO）→ [02 混合精度与显存账本](02-混合精度与显存账本.ipynb)
- 深度量化实现细节（GPTQ/AWQ/GGUF）→ [04-LLM大模型 22-深度量化GPTQ与AWQ](../../04-LLM大模型/教学/22-深度量化GPTQ与AWQ.ipynb)
- 专家分页 / 卸载 / 预取（量化如何降低专家搬运字节）→ [11 专家分页](11-专家分页与显存池化.ipynb) · [12 专家卸载](12-专家卸载与CPU-GPU协同.ipynb) · [13 预测式专家预取](13-预测式专家预取.ipynb)
- KV cache 的内存与分层（FP4 KV 的另一半故事）→ [05 KV Cache 与推理内存](05-KV%20Cache与推理内存.ipynb) · [14 KV Cache 分层与页级调度](14-KV%20Cache分层与页级调度.ipynb)
- 推理引擎与投机解码（低位宽 + 投机叠加）→ [06 推理引擎](06-推理引擎vLLM连续批处理与投机解码.ipynb)